# Machine Learning (DAT-B-ML)

# **Data Project**

## Organization

You should work on this project as a group of **up to three students** and upload your results to the dedicated upload folder in GRIPS. Participation in this project, including the **presentation for Assignment 1**, is required for bonus points in the exam.

During this project, there will be no homework sheets. Instead, your group can discuss progress, ask questions, and get advice during the meetings of your exercise group. In the final meetings of the semester, you will present your solution for Assignment 1.

*Based on the submissions, we will decide a winning team for the data project, which will receive a price.*

## Assignment 1
**Perform a supervised learning task on a a dataset** of your choice with the methods discussed in the lecture and **write a report** about your analysis. Additionally, you should **prepare a 5-minute presentation about your project for the meeting of your exercise group**.

*The dataset you select cannot be one of those used in the lecture.* Large databases for datasets are [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/), [Kaggle](https://www.kaggle.com/datasets), and [Hugging Face](https://huggingface.co/datasets), but you can also use other sources.

Pay attention to the **report guidelines** discussed in the central tutorial.

The final report must be in `PDF` format and should be uploaded as a `.zip` file together with your (reproducible) analysis preferably as a Jupyter notebook.

## Assignment 2

For this assignment, we provide an anonymized data set of WG room (shared-flat) listings in **Regensburg**. You will train a regression model to predict the **Kaltmiete** of each listing and a classification model to predict whether a listing is **overpriced**.

The data come in two files:

- `data/train.csv` — features for 1500 listings, plus the two label columns:
  - `kaltmiete_listing` — the listed Kaltmiete (regression label, **as a string**, see below).
  - `overpriced` — boolean, the classification label (`True` if the listing's €/m² is above some threshold).
- `data/test.csv` — features for 500 listings, **without labels**. Your task is to predict both labels for these listings.

> **Important:** the dataset is in a messy state. Some columns require parsing or cleaning before they are usable; some columns are unrelated to the targets; some pairs of columns are redundant or one is strictly less informative than the other. Also check for unrealistic outlier samples and remove them if appropriate. Treat data exploration and preprocessing as the first task — the off-the-shelf model on the raw frame will not be competitive.

### How to recover the numeric Kaltmiete

The `kaltmiete_listing` column comes in five formats. **The numeric Kaltmiete is always exactly recoverable** from each row, sometimes with help from the `nebenkosten_eur` column:

| Listing string             | Numeric Kaltmiete                    |
|----------------------------|--------------------------------------|
| `"450"`                    | 450                                  |
| `"450 €"`                  | 450                                  |
| `"450,00 EUR"`             | 450                                  |
| `"Kaltmiete: 450 €"`, `"450 kalt"`, `"kalt 450€"` | 450 |
| `"550 warm"`               | 550 − `nebenkosten_eur` for that row |

An unlabeled bare price (e.g. `"450"`) is **always** Kaltmiete in EUR. The suffix `warm` is the only signal that you must subtract Nebenkosten to recover Kaltmiete.

### Tasks and evaluation

We evaluate your predictions on the held-back true labels for the test set, using `r2_score` for regression and `accuracy_score` for classification. The team with the largest sum of these two metrics wins.

#### Task 1 — Regression
Train a regression model that predicts the `kaltmiete` for the listings in `test.csv`. Save your predictions as a CSV file named `kaltmiete_prediction.csv` and upload to the dedicated location in GRIPS. The file should contain **one column** named `kaltmiete` with **numeric** entries.

#### Task 2 — Classification
Train a classification model that predicts whether each listing in `test.csv` is `overpriced`. Save your predictions as a CSV file named `overpriced_prediction.csv` and upload to the dedicated location in GRIPS. The file should contain **one column** named `overpriced` with **boolean** entries.

> Hint: feel free to use any technique discussed in the lecture or elsewhere, but you must be able to explain everything from code the methodology when asked. Some methods will not be directly applicable to the dataset, so you may have to modify the dataset first.

> Hint: you should use the helper functions below to check whether your CSV file is a valid submission. The files need to match the format of the example files `kaltmiete_prediction_example.csv` and `overpriced_prediction_example.csv` but with your model's predictions.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
train.head()

In [ ]:
print("train shape:", train.shape)
print("test  shape:", test.shape)
print()
print("columns:")
for c in train.columns:
    print(f"  {c:<28} {train[c].dtype}")

### Submission format validators

In [ ]:
def check_file_regression(file_path):
    """Check that a regression-prediction CSV is valid for submission to Task 1."""
    try:
        predictions = pd.read_csv(file_path)
    except FileNotFoundError:
        print("File not found or could not be loaded.")
        return False
    if predictions.shape[0] != 500:
        print(f"Expected 500 rows, got {predictions.shape[0]}.")
        return False
    if predictions.shape[1] != 1:
        print("The file should contain exactly one column.")
        return False
    if predictions.columns[0] != "kaltmiete":
        print("The column name should be 'kaltmiete'.")
        return False
    if not np.issubdtype(predictions["kaltmiete"].dtype, np.number):
        print("The values in the 'kaltmiete' column should be numeric.")
        return False
    print("The file is a valid submission for Task 1.")
    return True


def check_file_classification(file_path):
    """Check that a classification-prediction CSV is valid for submission to Task 2."""
    try:
        predictions = pd.read_csv(file_path)
    except FileNotFoundError:
        print("File not found or could not be loaded.")
        return False
    if predictions.shape[0] != 500:
        print(f"Expected 500 rows, got {predictions.shape[0]}.")
        return False
    if predictions.shape[1] != 1:
        print("The file should contain exactly one column.")
        return False
    if predictions.columns[0] != "overpriced":
        print("The column name should be 'overpriced'.")
        return False
    if not np.issubdtype(predictions["overpriced"].dtype, np.bool_):
        print("The values in the 'overpriced' column should be boolean.")
        return False
    print("The file is a valid submission for Task 2.")
    return True

In [ ]:
check_file_regression("data/kaltmiete_prediction_example.csv")
check_file_classification("data/overpriced_prediction_example.csv")